# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'
!rm -rf /kaggle/working/*

In [2]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 70.9 MB/s eta 0:00:00
dependencies ok


In [3]:
import json, os, zipfile, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)

TASK_ID = 'task020'
H = W = 30
FORBIDDEN_OPS = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
MODEL_PATH = Path(f'{TASK_ID}.onnx')
REPORT_PATH = Path(f'{TASK_ID}_verification_report.json')
SUBMISSION_PATH = Path('submission.zip')
# Path.cwd()

In [4]:
def find_task_json(task_id='task020'):
    candidates = [
        Path(f'{task_id}.json'),
        Path('/mnt/data') / f'{task_id}.json',
    ]
    root = Path('/kaggle/input')
    if root.exists():
        candidates.extend(root.rglob(f'{task_id}.json'))
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f'Could not find {task_id}.json')

TASK_PATH = find_task_json(TASK_ID)
print('Using task JSON:', TASK_PATH)
with open(TASK_PATH) as f:
    task = json.load(f)

raw = []
for split in ['train', 'test', 'arc-gen']:
    if split in task:
        for idx, ex in enumerate(task[split]):
            raw.append({
                'split': split,
                'idx': idx,
                'input': np.array(ex['input'], dtype=np.int64),
                'output': np.array(ex['output'], dtype=np.int64),
            })
visible = [e for e in raw if e['split'] != 'arc-gen']
arcgen = [e for e in raw if e['split'] == 'arc-gen']
print('visible:', len(visible), 'arc-gen:', len(arcgen), 'all:', len(raw))


Using task JSON: /kaggle/input/competitions/neurogolf-2026/task020.json
visible: 4 arc-gen: 262 all: 266


In [5]:
def grid_to_onehot(grid, H=30, W=30):
    arr = np.array(grid, dtype=np.int64)
    oh = np.zeros((10, H, W), dtype=np.float32)
    h, w = arr.shape
    for r in range(h):
        for c in range(w):
            oh[arr[r, c], r, c] = 1.0
    return oh

def onehot_to_grid(y, h, w):
    return y.argmax(0)[:h, :w].astype(np.int64)

def batch_onehot(examples):
    return np.stack([grid_to_onehot(ex['input']) for ex in examples]).astype(np.float32)


In [6]:
# Structural tags used for validation splitting.
# The task uses a 5x5 local lattice with three 4-slot classes plus a center.
SLOT_CLASSES = {
    'corners': [(-2, -2), (-2, 2), (2, -2), (2, 2)],
    'edge_mids': [(-2, 0), (0, -2), (0, 2), (2, 0)],
    'inner_diag': [(-1, -1), (-1, 1), (1, -1), (1, 1)],
}
ALL_SLOTS = SLOT_CLASSES['corners'] + SLOT_CLASSES['edge_mids'] + SLOT_CLASSES['inner_diag'] + [(0, 0)]

def structural_tags(ex):
    inp, out = ex['input'], ex['output']
    nz = np.argwhere(out > 0)
    r0, c0 = nz.min(0)
    r1, c1 = nz.max(0)
    cy, cx = (r0 + r1) // 2, (c0 + c1) // 2
    diff = np.argwhere(out != inp)
    fill_color = int(out[tuple(diff[0])]) if len(diff) else -1
    fill_class = 'unknown'
    if len(diff):
        first = tuple(diff[0])
        for name, offsets in SLOT_CLASSES.items():
            pts = {(cy + dr, cx + dc) for dr, dc in offsets}
            if first in pts:
                fill_class = name
    return {
        'bbox_top': int(r0),
        'bbox_left': int(c0),
        'bbox_bottom': int(r1),
        'bbox_right': int(c1),
        'center': [int(cy), int(cx)],
        'fill_color': fill_color,
        'fill_class': fill_class,
        'colors': [int(c) for c in sorted(np.unique(out)) if c != 0],
        'shape': list(inp.shape),
        'distractor': bool(np.sum(out > 0) != 13),
    }

for ex in raw:
    ex['tags'] = structural_tags(ex)

# Structural split: hold out composite fill-class/color/position keys.
keys = []
for ex in arcgen:
    t = ex['tags']
    keys.append((t['fill_class'], t['fill_color'], t['bbox_top'], t['bbox_left']))
unique_keys = sorted(set(keys))
holdout_keys = set(unique_keys[::3])
struct_holdout = [ex for ex in arcgen if (ex['tags']['fill_class'], ex['tags']['fill_color'], ex['tags']['bbox_top'], ex['tags']['bbox_left']) in holdout_keys]
target_n = round(0.30 * len(arcgen))
struct_holdout = struct_holdout[:target_n]
holdout_ids = {(ex['split'], ex['idx']) for ex in struct_holdout}
struct_train = [ex for ex in arcgen if (ex['split'], ex['idx']) not in holdout_ids]

print('structural train arc-gen:', len(struct_train))
print('structural holdout arc-gen:', len(struct_holdout))
print('holdout fill classes:', sorted(set(ex['tags']['fill_class'] for ex in struct_holdout)))
print('holdout fill colors:', sorted(set(ex['tags']['fill_color'] for ex in struct_holdout)))
print('holdout bbox positions:', len(set((ex['tags']['bbox_top'], ex['tags']['bbox_left']) for ex in struct_holdout)))


structural train arc-gen: 183
structural holdout arc-gen: 79
holdout fill classes: ['corners', 'edge_mids', 'inner_diag']
holdout fill colors: [1, 2, 3, 4, 8]
holdout bbox positions: 15


In [7]:
def build_slot_kernels():
    cases = []
    for class_name, offsets in SLOT_CLASSES.items():
        for target in offsets:
            cases.append((class_name, target, offsets))
    K = len(cases)
    kernel_size = 9
    center = kernel_size // 2
    same_kernels = []
    class_kernels = []
    union_kernels = []
    for class_name, target, offsets in cases:
        k_same = torch.zeros(1, kernel_size, kernel_size)
        k_class = torch.zeros(1, kernel_size, kernel_size)
        k_union = torch.zeros(1, kernel_size, kernel_size)
        for dr, dc in offsets:
            if (dr, dc) == target:
                continue
            rr, cc = dr - target[0] + center, dc - target[1] + center
            if 0 <= rr < kernel_size and 0 <= cc < kernel_size:
                k_same[0, rr, cc] = 1.0
                k_class[0, rr, cc] = 1.0
        for dr, dc in ALL_SLOTS:
            if (dr, dc) == target:
                continue
            rr, cc = dr - target[0] + center, dc - target[1] + center
            if 0 <= rr < kernel_size and 0 <= cc < kernel_size:
                k_union[0, rr, cc] = 1.0
        same_kernels.append(k_same)
        class_kernels.append(k_class)
        union_kernels.append(k_union)
    return torch.stack(same_kernels, 0), torch.stack(class_kernels, 0), torch.stack(union_kernels, 0)

SAME_BASE, CLASS_BASE, UNION_BASE = build_slot_kernels()
K = SAME_BASE.shape[0]
print('slot target cases:', K)


slot target cases: 12


In [8]:
class ResidualSlotCompletionCNN(nn.Module):
    '''
    Residual slot-completion model.

    It builds local-lattice candidate features using fixed convolution kernels.
    Existing nonzero input pixels are frozen; only zero/background cells may be changed.
    The support_threshold is selected using the structural train split, then evaluated on the held-out split.
    '''
    def __init__(self, support_threshold=10.0):
        super().__init__()
        self.support_threshold = float(support_threshold)
        self.register_buffer('same_w', SAME_BASE.repeat(9, 1, 1, 1))
        self.register_buffer('class_w', CLASS_BASE)
        self.register_buffer('union_w', UNION_BASE)
        yy = torch.linspace(-1, 1, H).view(1, 1, H, 1).expand(1, 1, H, W)
        xx = torch.linspace(-1, 1, W).view(1, 1, 1, W).expand(1, 1, H, W)
        self.register_buffer('coord_y', yy)
        self.register_buffer('coord_x', xx)

    def forward(self, x):
        colors = x[:, 1:10]
        valid = (x.sum(dim=1, keepdim=True) > 0.5).float()
        existing = (colors.sum(dim=1, keepdim=True) > 0.5).float()
        empty = ((x[:, 0:1] > 0.5).float()) * valid * (1.0 - existing)
        nonzero = colors.sum(dim=1, keepdim=True)

        # Candidate local-lattice features.
        same = F.conv2d(colors, self.same_w, padding=4, groups=9)
        class_nz = F.conv2d(nonzero, self.class_w, padding=4)
        union_nz = F.conv2d(nonzero, self.union_w, padding=4)

        B = x.shape[0]
        same = same.view(B, 9, K, H, W)

        same_present = (same > 0.5).float()
        no_other_color = ((class_nz.unsqueeze(1) - same).abs() < 0.5).float()
        motif_support = (union_nz.unsqueeze(1) >= (self.support_threshold - 0.5)).float()
        candidate = (same_present * no_other_color * motif_support).amax(dim=2)
        candidate = candidate * empty

        add_any = (candidate.sum(dim=1, keepdim=True) > 0.5).float()
        background = (1.0 - add_any) * (1.0 - existing) * valid
        additive_output = torch.cat([background, candidate], dim=1)
        output = existing * x + (1.0 - existing) * additive_output
        return output * valid


In [9]:
def evaluate_torch(model, examples):
    if not examples:
        return {'right': 0, 'total': 0, 'wrong': 0}
    xs = batch_onehot(examples)
    with torch.no_grad():
        ys = model(torch.tensor(xs, dtype=torch.float32)).cpu().numpy()
    wrong = 0
    for ex, y in zip(examples, ys):
        pred = onehot_to_grid(y, *ex['output'].shape)
        if not np.array_equal(pred, ex['output']):
            wrong += 1
    return {'right': len(examples) - wrong, 'total': len(examples), 'wrong': wrong}

# Select the support threshold on the structural train split only.
threshold_trials = [7.0, 8.0, 9.0, 10.0, 11.0, 12.0]
selection = []
for th in threshold_trials:
    m = ResidualSlotCompletionCNN(support_threshold=th).eval()
    train_score = evaluate_torch(m, visible + struct_train)
    selection.append({'threshold': th, **train_score})
selection


[{'threshold': 7.0, 'right': 0, 'total': 187, 'wrong': 187},
 {'threshold': 8.0, 'right': 125, 'total': 187, 'wrong': 62},
 {'threshold': 9.0, 'right': 187, 'total': 187, 'wrong': 0},
 {'threshold': 10.0, 'right': 187, 'total': 187, 'wrong': 0},
 {'threshold': 11.0, 'right': 0, 'total': 187, 'wrong': 187},
 {'threshold': 12.0, 'right': 0, 'total': 187, 'wrong': 187}]

In [10]:
# Choose the first threshold that passes visible + structural-train examples.
passing = [row for row in selection if row['right'] == row['total']]
assert passing, selection
SELECTED_THRESHOLD = passing[0]['threshold']
print('selected support threshold:', SELECTED_THRESHOLD)
model = ResidualSlotCompletionCNN(support_threshold=SELECTED_THRESHOLD).eval()

validation = {
    'visible': evaluate_torch(model, visible),
    'arcgen_structural_train': evaluate_torch(model, struct_train),
    'arcgen_structural_holdout': evaluate_torch(model, struct_holdout),
    'arcgen_all': evaluate_torch(model, arcgen),
    'all_provided': evaluate_torch(model, raw),
}
validation


selected support threshold: 9.0


{'visible': {'right': 4, 'total': 4, 'wrong': 0},
 'arcgen_structural_train': {'right': 183, 'total': 183, 'wrong': 0},
 'arcgen_structural_holdout': {'right': 79, 'total': 79, 'wrong': 0},
 'arcgen_all': {'right': 262, 'total': 262, 'wrong': 0},
 'all_provided': {'right': 266, 'total': 266, 'wrong': 0}}

In [11]:
# Export ONNX. The public interface is fixed to [1,10,30,30].
dummy = torch.zeros(1, 10, 30, 30, dtype=torch.float32)
torch.onnx.export(
    model,
    dummy,
    str(MODEL_PATH),
    input_names=['input'],
    output_names=['output'],
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)
print('exported:', MODEL_PATH, 'size:', MODEL_PATH.stat().st_size)


/tmp/ipykernel_16/4024884130.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


exported: task020.onnx size: 48031


In [12]:
import onnx
import onnxruntime as ort

onnx_model = onnx.load(str(MODEL_PATH))
onnx.checker.check_model(onnx_model)
ops = sorted(set(node.op_type for node in onnx_model.graph.node))
forbidden = sorted(set(ops) & FORBIDDEN_OPS)

def shape_of(value_info):
    dims = []
    for dim in value_info.type.tensor_type.shape.dim:
        dims.append(dim.dim_value if dim.dim_value else dim.dim_param)
    return dims

input_shape = shape_of(onnx_model.graph.input[0])
output_shape = shape_of(onnx_model.graph.output[0])
print('input shape:', input_shape)
print('output shape:', output_shape)
print('forbidden ops:', forbidden)
assert input_shape == [1, 10, 30, 30]
assert output_shape == [1, 10, 30, 30]
assert not forbidden
assert MODEL_PATH.stat().st_size < 1_400_000


input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
forbidden ops: []


In [13]:
def evaluate_onnx(examples):
    if not examples:
        return {'right': 0, 'total': 0, 'wrong': 0}
    sess = ort.InferenceSession(str(MODEL_PATH), providers=['CPUExecutionProvider'])
    input_name = sess.get_inputs()[0].name
    wrong = 0
    for ex in examples:
        x = grid_to_onehot(ex['input'])[None].astype(np.float32)
        y = sess.run(None, {input_name: x})[0][0]
        pred = onehot_to_grid(y, *ex['output'].shape)
        if not np.array_equal(pred, ex['output']):
            wrong += 1
    return {'right': len(examples) - wrong, 'total': len(examples), 'wrong': wrong}

onnx_validation = {
    'visible': evaluate_onnx(visible),
    'arcgen_structural_train': evaluate_onnx(struct_train),
    'arcgen_structural_holdout': evaluate_onnx(struct_holdout),
    'arcgen_all': evaluate_onnx(arcgen),
    'all_provided': evaluate_onnx(raw),
}
onnx_validation


{'visible': {'right': 4, 'total': 4, 'wrong': 0},
 'arcgen_structural_train': {'right': 183, 'total': 183, 'wrong': 0},
 'arcgen_structural_holdout': {'right': 79, 'total': 79, 'wrong': 0},
 'arcgen_all': {'right': 262, 'total': 262, 'wrong': 0},
 'all_provided': {'right': 266, 'total': 266, 'wrong': 0}}

In [14]:
report = {
    'task': TASK_ID,
    'model': 'Residual slot-completion CNN with coordinate buffers, fixed local-lattice candidate convolution features, and frozen input cells',
    'selected_support_threshold': SELECTED_THRESHOLD,
    'selection_on_visible_plus_structural_train': selection,
    'validation_protocol': {
        'split': 'structural composite-key split, not random',
        'structural_key': 'fill_class/fill_color/bbox_top/bbox_left',
        'arcgen_train': len(struct_train),
        'arcgen_holdout': len(struct_holdout),
        'holdout_fill_classes': sorted(set(ex['tags']['fill_class'] for ex in struct_holdout)),
        'holdout_fill_colors': sorted(set(ex['tags']['fill_color'] for ex in struct_holdout)),
        'holdout_bbox_positions': len(set((ex['tags']['bbox_top'], ex['tags']['bbox_left']) for ex in struct_holdout)),
    },
    'torch_validation': validation,
    'onnx_validation': onnx_validation,
    'onnx': {
        'file': str(MODEL_PATH),
        'size_bytes': MODEL_PATH.stat().st_size,
        'input_shape': input_shape,
        'output_shape': output_shape,
        'ops': ops,
        'forbidden_ops': forbidden,
    },
}
with open(REPORT_PATH, 'w') as f:
    json.dump(report, f, indent=2)

with zipfile.ZipFile(SUBMISSION_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(MODEL_PATH, arcname='task020.onnx')

print('wrote:', REPORT_PATH)
print('wrote:', SUBMISSION_PATH)


wrote: task020_verification_report.json
wrote: submission.zip
